# Fitting Parameters and Retrievals

TauREx models dynamically expose their adjustable parameters through the `fittingParameters` dictionary. An optimizer can then be connected to the model and an observation to perform atmospheric retrievals.

This example covers:
1. Listing and modifying fitting parameters on a model.
2. Creating a synthetic observation from a forward model.
3. Running a nested-sampling retrieval with `NestleOptimizer`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from _shared import build_transmission_model

context = build_transmission_model(include_cia=True, include_rayleigh=True, download=False)
tm = context['tm']

## Fitting Parameters

The set of available fitting parameters depends on which components are attached to the model. Swapping a temperature profile (e.g., `Isothermal` vs. `Guillot2010`) changes which parameters appear.

In [ ]:
print('Available fitting parameters:')
for name in sorted(tm.fittingParameters.keys()):
    print(f'  {name}')

Parameters can be read and written through the bracket operator on the model.

In [ ]:
print(f'Current T: {tm["T"]}')
print(f'Current H2O: {tm["H2O"]}')

# Modify a parameter
tm['T'] = 1800.0
print(f'Updated T: {tm["T"]}')

# Reset
tm['T'] = 2000.0

## Creating a Synthetic Observation

To demonstrate a retrieval without requiring external data files, a synthetic observation is generated from the forward model and perturbed with Gaussian noise. The `ArraySpectrum` class wraps the resulting array into an object the optimizer can consume.

In [ ]:
from taurex.binning import SimpleBinner
from taurex.data.spectrum.array import ArraySpectrum

# Generate a binned "true" spectrum
obs_wl = np.logspace(np.log10(0.8), np.log10(10.0), 50)
obs_wn = np.sort(10000 / obs_wl)
binner = SimpleBinner(wngrid=obs_wn)
bin_wn, bin_rprs, _, _ = binner.bin_model(tm.model(wngrid=obs_wn))
bin_wl = 10000 / bin_wn[::-1]
bin_rprs = bin_rprs[::-1]

# Add noise
rng = np.random.default_rng(42)
noise_level = 3e-5
noisy_rprs = bin_rprs + rng.normal(0, noise_level, size=bin_rprs.shape)
errors = np.full_like(bin_rprs, noise_level)

# Package as an observation (columns: wavelength, data, error)
obs_array = np.column_stack([bin_wl, noisy_rprs, errors])
obs = ArraySpectrum(spectrum=obs_array)

print(f'Synthetic observation: {len(bin_wl)} bins, noise = {noise_level:.1e}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.errorbar(obs.wavelengthGrid, obs.spectrum, obs.errorBar,
             fmt='.', capsize=2, label='Synthetic observation')
plt.plot(bin_wl, bin_rprs, lw=2, label='True model')
plt.xscale('log')
plt.xlabel('Wavelength (um)')
plt.ylabel('$(R_p/R_s)^2$')
plt.title('Synthetic observation with noise')
plt.legend()
plt.grid(alpha=0.2)

## Setting Up the Retrieval

`NestleOptimizer` performs nested sampling using the [nestle](http://kylebarbary.com/nestle/) library. The optimizer needs:
1. A forward model (`set_model`).
2. An observation (`set_observed`).
3. At least one enabled fitting parameter with defined boundaries (`enable_fit`, `set_boundary`).

In [ ]:
from taurex.optimizer.nestle import NestleOptimizer

opt = NestleOptimizer(num_live_points=50)
opt.set_model(tm)
opt.set_observed(obs)

opt.enable_fit('planet_radius')
opt.set_boundary('planet_radius', [1.0, 1.8])

opt.enable_fit('T')
opt.set_boundary('T', [1200, 2800])

print('Fitting parameters:')
for name in opt.fit_names:
    print(f'  {name}: bounds = {opt.fit_boundaries[opt.fit_names.index(name)]}')

## Running the Retrieval

Calling `fit()` runs the nested sampler. With only 50 live points and 2 free parameters this completes quickly; production runs use more live points for better posterior sampling.

In [ ]:
import taurex.log
taurex.log.disableLogging()

solution = opt.fit()

print('Retrieval complete.')

## Inspecting the Results

The `get_solution()` generator yields each solution mode. For each solution, `update_model` sets the model to the best-fit values so the corresponding spectrum can be plotted against the observation.

In [ ]:
obs_binner = obs.create_binner()

for sol, optimized_map, optimized_value, values in opt.get_solution():
    opt.update_model(optimized_map)

    print('Best-fit values:')
    for name, value in zip(opt.fit_names, optimized_value):
        print(f'  {name} = {value:.4f}')

    fit_wn, fit_rprs, _, _ = obs_binner.bin_model(tm.model(obs.wavenumberGrid))
    fit_wl = 10000 / fit_wn[::-1]
    fit_rprs = fit_rprs[::-1]

    plt.figure(figsize=(7, 4))
    plt.errorbar(obs.wavelengthGrid, obs.spectrum, obs.errorBar,
                 fmt='.', capsize=2, label='Observation')
    plt.plot(fit_wl, fit_rprs, lw=2, label='Best fit')
    plt.xscale('log')
    plt.xlabel('Wavelength (um)')
    plt.ylabel('$(R_p/R_s)^2$')
    plt.title('Retrieval result')
    plt.legend()
    plt.grid(alpha=0.2)